# Half of it comes back: fashion returns

Stephen tAMBI

Online fashion retailers like Zalando say about half of ordered items are returned. This notebook uses 2.3 million order lines from a German online fashion retailer (Data Mining Cup 2016, prudsys AG) to find what drives returns, predict them at checkout and price what cutting them is worth.

Place `orders_train.txt` in `../data/`.

In [1]:
import pandas as pd, numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('../data/orders_train.txt', sep=';')
df = df[(df.quantity > 0) & (df.price > 0) & df.rrp.notna() & df.productGroup.notna()].copy()
df['orderDate'] = pd.to_datetime(df.orderDate)
print(f'{len(df):,} order lines, {df.orderID.nunique():,} orders, {df.customerID.nunique():,} customers')
print(f'Item return rate: {df.returnQuantity.sum() / df.quantity.sum():.1%}')
print(f'Value returned: €{(df.price * df.returnQuantity / df.quantity).sum():,.0f} of €{df.price.sum():,.0f}')

2,277,971 order lines, 738,376 orders, 311,165 customers
Item return rate: 52.7%
Value returned: €46,786,251 of €82,291,025


## Drivers known at checkout

In [2]:
df['sizes'] = np.minimum(df.groupby(['orderID', 'articleID', 'colorCode']).sizeCode.transform('nunique'), 3)
df['colors'] = np.minimum(df.groupby(['orderID', 'articleID']).colorCode.transform('nunique'), 3)
df['orderItems'] = df.groupby('orderID').quantity.transform('sum')
df['disc'] = 1 - df.price / df.rrp
df['payLater'] = df.paymentMethod.isin(['BPRG', 'RG', 'KGRG', 'BPPL'])

def rate(col):
    g = df.groupby(col, observed=True).agg(items=('quantity', 'sum'), returned=('returnQuantity', 'sum'))
    g['share_%'] = (g['items'] / g['items'].sum() * 100).round(1)
    g['return_rate_%'] = (g.returned / g['items'] * 100).round(1)
    return g[['share_%', 'return_rate_%']]

rate('sizes')

,share_%,return_rate_%
sizes,,
1,86.4,49.0
2,13.2,75.9
3,0.4,73.6


In [3]:
rate(np.minimum(df.orderItems, 6).rename('items_in_basket'))

,share_%,return_rate_%
items_in_basket,,
1,8.7,36.1
2,16.7,43.5
3,15.7,47.0
4,14.0,52.8
5,10.7,56.0
6,34.2,62.8


In [4]:
rate(pd.cut(df.price, [0, 15, 30, 50, 80, 1e5]).rename('price'))

,share_%,return_rate_%
price,,
"(0.0, 15.0]",13.2,40.1
"(15.0, 30.0]",40.3,49.0
"(30.0, 50.0]",29.9,58.0
"(50.0, 80.0]",14.0,61.6
"(80.0, 100000.0]",2.6,63.4


In [5]:
rate(df.payLater.rename('pay_later'))

,share_%,return_rate_%
pay_later,,
False,22.0,39.7
True,78.0,56.3


## Who returns

In [6]:
cust = df.groupby('customerID').returnQuantity.sum().sort_values(ascending=False)
cum = cust.cumsum() / cust.sum()
print(f'Top 10% of customers: {cum.iloc[int(len(cust) * .1)]:.1%} of returns')
print(f'Customers who never returned: {(cust == 0).mean():.1%}')

Top 10% of customers: 53.5% of returns
Customers who never returned: 31.3%


## Model: will this item come back?
Customer history uses only earlier days. Train Jan 2014 to Mar 2015, test Apr to Sep 2015.

In [7]:
day = df.groupby(['customerID', 'orderDate']).agg(q=('quantity', 'sum'), r=('returnQuantity', 'sum')).reset_index().sort_values(['customerID', 'orderDate'])
day['histQ'] = day.groupby('customerID').q.cumsum() - day.q
day['histR'] = day.groupby('customerID').r.cumsum() - day.r
df = df.merge(day[['customerID', 'orderDate', 'histQ', 'histR']], on=['customerID', 'orderDate'])
df['histRate'] = np.where(df.histQ > 0, df.histR / df.histQ.clip(lower=1), np.nan)

feats = ['price', 'rrp', 'disc', 'sizes', 'colors', 'orderItems', 'payLater', 'deviceID', 'productGroup', 'voucherAmount', 'histQ', 'histRate']
X, y = df[feats].astype(float), (df.returnQuantity > 0).astype(int)
tr, te = df.orderDate < '2015-04-01', df.orderDate >= '2015-04-01'
m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, max_leaf_nodes=63, random_state=42).fit(X[tr], y[tr])
p = m.predict_proba(X[te])[:, 1]
print('ROC-AUC', round(roc_auc_score(y[te], p), 3))
pd.Series(y[te].to_numpy()).groupby(pd.qcut(-p, 10, labels=range(1, 11))).mean().mul(100).round(1).rename('actual return rate % by risk decile (1 = riskiest)')

ROC-AUC 0.73


1     81.8
2     74.7
3     69.4
4     64.5
5     58.8
6     52.7
7     46.3
8     38.6
9     29.6
10    15.3
Name: actual return rate % by risk decile (1 = riskiest), dtype: float64

## Recommendations
1. **Size advice** when a second size of the same item goes into the basket (76% of bracketed items come back).
2. **Use the risk score to help:** extra fit information on the riskiest items only.
3. **Target extreme returners,** not everyone: 10% of customers make over half of all returns.
4. **Keep invoice payment** but use it as a model signal.

Interactive version: https://stambi292.github.io/fashion-returns/